# Oscar 1927–2024: cinco perguntas estratégicas

Este notebook consome o pacote `projeto1_c11`. A lógica de cada pergunta vive em `src/projeto1_c11/questions/`; aqui entram a narrativa, as tabelas e os gráficos do pitch.

Dataset: [`unanimad/the-oscar-award`](https://www.kaggle.com/datasets/unanimad/the-oscar-award).

## 0. Carga e descoberta

In [ ]:
from projeto1_c11.data import load_oscars
from projeto1_c11.plotting import apply_style, save_fig
from projeto1_c11.questions import (
    q01_conversao,
    q02_precursoras,
    q03_evolucao,
    q04_eficiencia,
    q05_concentracao,
)

apply_style()
df = load_oscars()
df.head()

In [ ]:
print(f'linhas={len(df):,}  colunas={df.shape[1]}')
print(df.dtypes)
df.isna().sum().to_frame('n_ausentes')

In [ ]:
n_raw = df['category'].nunique()
n_canon = df['category_canon'].nunique()
print(f'categorias originais={n_raw}  após unificar Melhor Filme={n_canon}')

remakes = (
    df.dropna(subset=['film'])
    .groupby('film')['year_ceremony']
    .nunique()
    .loc[lambda s: s > 1]
    .sort_values(ascending=False)
)
print(f'títulos em mais de uma cerimônia={len(remakes)}')
remakes.head(10).to_frame('n_cerimonias')

### Três desafios da etapa de Descoberta (Slide 4)

1. **Padronização de categorias históricas.** Melhor Filme já se chamou *Outstanding Picture*, *Outstanding Production*, *Outstanding Motion Picture* e *Best Motion Picture*. Sem unificar esses nomes, qualquer análise da categoria principal parte o século no meio.
2. **Valores ausentes.** Prêmios honorários e especiais frequentemente vêm sem `film` (e às vezes sem `name` padronizado). Por isso as perguntas de filme/profissional excluem `is_honorary` e descartam nulos da chave.
3. **Filmes homônimos.** Remakes compartilham o título; a chave de filme é sempre `['film', 'year_ceremony']`, nunca só o nome.

## Q1. Volume de indicações e conversão em vitórias

Qual é a relação entre o volume de indicações de um filme e sua taxa de conversão em vitórias? Existe um ponto de virada?

**Como lemos:** cada ponto é um filme-cerimônia. O painel da direita mostra a taxa média de vitória por faixa de indicações — o candidato a "ponto de virada" para o investimento em campanha FYC.

In [ ]:
print(q01_conversao.QUESTION)
q01 = q01_conversao.compute(df)
display(q01_conversao.correlations(q01).to_frame('correlação'))
display(q01_conversao.conversion_by_bin(q01))
q01.head(10)

In [ ]:
fig = q01_conversao.plot(q01)
save_fig(fig, q01_conversao.SLUG)
fig

**Insight.** Mais indicações acompanham mais vitórias (correlação positiva), mas a conversão não é linear: o salto de faixa costuma aparecer nas faixas altas, enquanto o extremo direito também concentra os maiores esnobados (muitas indicações e zero estatuetas). Para o estúdio, pulverizar campanha em dezenas de categorias não garante a noite — e pode produzir o pior dos mundos: visibilidade máxima sem a foto da estatueta.

## Q2. Categorias precursoras de Melhor Filme

Para um filme vencer Melhor Filme, quais outras categorias historicamente têm a maior correlação de vitória conjunta?

Universo: filmes **indicados** a Melhor Filme (não a indústria inteira). Para cada categoria X: `P(venceu X | venceu BP)`, `P(venceu BP | venceu X)` e o coeficiente φ entre as duas binárias.

In [ ]:
print(q02_precursoras.QUESTION)
q02 = q02_precursoras.compute(df)
q02.head(15)

In [ ]:
fig = q02_precursoras.plot(q02)
save_fig(fig, q02_precursoras.SLUG)
fig

**Insight.** Direção, montagem e roteiro original/adaptado costumam ser o termômetro real da noite: um filme que leva essas categorias raramente é só "indicado de volume". Produtoras e apostadores devem tratar essas estatuetas como precursoras, não como prêmios consolação.

## Q3. Evolução da Academia por década

Como o perfil da Academia evoluiu ao longo das décadas em termos de diversidade de categorias e quantidade de prêmios distribuídos por cerimônia?

In [ ]:
print(q03_evolucao.QUESTION)
q03 = q03_evolucao.compute(df)
q03

In [ ]:
fig = q03_evolucao.plot(q03)
save_fig(fig, q03_evolucao.SLUG)
fig

**Insight.** A Academia do século XXI reconhece mais ofícios do que a dos anos 1930: efeitos, som, animação, documentário. Isso abre frentes de prestígio (e de campanha) que não existiam quando o prêmio era quase só filme, direção e atuação.

## Q4. Volume vs. eficiência de profissionais

Quais profissionais ou entidades possuem o maior histórico de indicações e quem tem a melhor eficiência entre os mais indicados?

Filtro: pelo menos 5 indicações competitivas, excluindo filme internacional (onde `name` é o país). Dois rankings — quem mais é chamado à festa e quem melhor converte o convite em estatueta.

In [ ]:
print(q04_eficiencia.QUESTION)
q04 = q04_eficiencia.compute(df)
display(q04.head(15))
q04.sort_values(['win_rate', 'nominations'], ascending=False).head(15)

In [ ]:
fig = q04_eficiencia.plot(q04)
save_fig(fig, q04_eficiencia.SLUG)
fig

**Insight.** *Star power* compra indicação com mais previsibilidade do que vitória. Contratar o profissional mais indicado aumenta a chance de entrar na cédula; a taxa de conversão dos recordistas costuma ser bem mais baixa do que o marketing de "vencedor do Oscar" sugere. Eficiência e volume não são a mesma lista.

## Q5. Concentração vs. pulverização da noite

Qual é o nível de concentração vs. democratização do Oscar ao longo do tempo: quantos filmes diferentes dividem os prêmios em uma cerimônia?

Por cerimônia: quantos filmes distintos levaram pelo menos uma estatueta competitiva, e que fração dos prêmios ficou com o filme mais premiado (efeito varredura).

In [ ]:
print(q05_concentracao.QUESTION)
q05 = q05_concentracao.compute(df)
display(q05.tail(10))
from projeto1_c11.questions.q05_concentracao import decade_means
decade_means(q05)

In [ ]:
fig = q05_concentracao.plot(q05)
save_fig(fig, q05_concentracao.SLUG)
fig

**Insight.** Noites de varredura (um filme levando fatia grande do quadro) aumentam o risco para concorrentes: o prestígio e a narrativa da imprensa ficam monopolizados. Décadas mais pulverizadas abrem espaço para independentes e *streamings* saírem com 1–2 estatuetas e ainda assim venderem o selo Oscar.